In [1]:
import os
import math
import itertools
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# =========================
# 可选：解决中文显示问题
# 如果你的环境没有 SimHei，可以改成 Microsoft YaHei、Arial Unicode MS 等
# =========================
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


# =========================
# 输出目录
# =========================
OUTPUT_DIR = "/workspace/data/13_sections"
# os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# 立方体基础定义
# 立方体采用 [0,1]^3
# =========================
CUBE_CENTER = np.array([0.5, 0.5, 0.5], dtype=float)

# 8 个顶点
VERTICES = np.array(list(itertools.product([0.0, 1.0], repeat=3)), dtype=float)

# 12 条边（两个顶点坐标只有一个维度不同）
EDGES = []
for i in range(len(VERTICES)):
    for j in range(i + 1, len(VERTICES)):
        if np.sum(np.abs(VERTICES[i] - VERTICES[j])) == 1.0:
            EDGES.append((i, j))


# =========================
# 13 个方向
# normal: 切面的法向量
# =========================
DIRECTIONS = [
    # 横矢冠 3 个
    {"id": 1, "name": "横向（上下方向）", "normal": (0, 0, 1), "group": "横矢冠", "color": "#4CAF50"},
    {"id": 2, "name": "矢状向（左右方向）", "normal": (1, 0, 0), "group": "横矢冠", "color": "#1E88E5"},
    {"id": 3, "name": "冠状向（前后方向）", "normal": (0, 1, 0), "group": "横矢冠", "color": "#8E44AD"},

    # 体对角方向 4 个
    {"id": 4, "name": "体对角方向 1", "normal": (-1, -1, 2), "group": "体对角", "color": "#E53935"},
    {"id": 5, "name": "体对角方向 2", "normal": (1, 1, 2), "group": "体对角", "color": "#FB8C00"},
    {"id": 6, "name": "体对角方向 3", "normal": (1, -1, -2), "group": "体对角", "color": "#43A047"},
    {"id": 7, "name": "体对角方向 4", "normal": (-1, 1, -2), "group": "体对角", "color": "#3949AB"},

    # 面对角方向 6 个
    {"id": 8,  "name": "面对角方向 1", "normal": (1, 1, 0), "group": "面对角", "color": "#D81B60"},
    {"id": 9,  "name": "面对角方向 2", "normal": (1, -1, 0), "group": "面对角", "color": "#FBC02D"},
    {"id": 10, "name": "面对角方向 3", "normal": (1, 0, 1), "group": "面对角", "color": "#00ACC1"},
    {"id": 11, "name": "面对角方向 4", "normal": (1, 0, -1), "group": "面对角", "color": "#5E35B1"},
    {"id": 12, "name": "面对角方向 5", "normal": (0, 1, 1), "group": "面对角", "color": "#8D6E63"},
    {"id": 13, "name": "面对角方向 6", "normal": (0, 1, -1), "group": "面对角", "color": "#7CB342"},
]


# =========================
# 工具函数
# =========================
def normalize(v):
    v = np.asarray(v, dtype=float)
    norm = np.linalg.norm(v)
    if norm == 0:
        raise ValueError("零向量不能归一化")
    return v / norm


def unique_points(points, tol=1e-8):
    """
    点去重
    """
    uniq = []
    for p in points:
        p = np.asarray(p, dtype=float)
        if not any(np.linalg.norm(p - q) < tol for q in uniq):
            uniq.append(p)
    return uniq


def plane_cube_intersection(center, normal, vertices, edges, tol=1e-9):
    """
    计算平面与立方体的交多边形顶点
    平面方程：normal · (x - center) = 0
    """
    n = normalize(normal)
    points = []

    for i, j in edges:
        p0 = vertices[i]
        p1 = vertices[j]
        d = p1 - p0

        f0 = np.dot(n, p0 - center)
        f1 = np.dot(n, p1 - center)

        # 整条边都在平面内
        if abs(f0) < tol and abs(f1) < tol:
            points.append(p0)
            points.append(p1)
            continue

        # 一个端点在平面上
        if abs(f0) < tol:
            points.append(p0)
        if abs(f1) < tol:
            points.append(p1)

        # 两端异号，说明有交点
        if f0 * f1 < -tol * tol:
            t = f0 / (f0 - f1)   # p = p0 + t*(p1-p0)
            p = p0 + t * d
            points.append(p)

    points = unique_points(points, tol=1e-7)

    if len(points) < 3:
        raise RuntimeError("交点少于 3 个，无法形成切面多边形")

    # 对点按平面内角度排序
    pts = np.array(points)
    centroid = np.mean(pts, axis=0)

    # 在平面内构造两个正交基 e1, e2
    tmp = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(tmp, n)) > 0.9:
        tmp = np.array([0.0, 1.0, 0.0])

    e1 = np.cross(n, tmp)
    e1 = normalize(e1)
    e2 = np.cross(n, e1)
    e2 = normalize(e2)

    angles = []
    for p in pts:
        v = p - centroid
        x = np.dot(v, e1)
        y = np.dot(v, e2)
        angles.append(math.atan2(y, x))

    order = np.argsort(angles)
    return pts[order]


def is_visible_edge(v0, v1):
    """
    简单定义“可见边”：
    位于前面(y=0)、右面(x=1)、上面(z=1)的边画实线；
    其他画虚线。
    """
    def same_face(a, b, axis, value):
        return abs(a[axis] - value) < 1e-9 and abs(b[axis] - value) < 1e-9

    if same_face(v0, v1, 1, 0.0):  # front face
        return True
    if same_face(v0, v1, 0, 1.0):  # right face
        return True
    if same_face(v0, v1, 2, 1.0):  # top face
        return True
    return False


def draw_cube(ax):
    """
    画立方体框架
    """
    for i, j in EDGES:
        p0 = VERTICES[i]
        p1 = VERTICES[j]

        if is_visible_edge(p0, p1):
            ax.plot(
                [p0[0], p1[0]],
                [p0[1], p1[1]],
                [p0[2], p1[2]],
                color="black",
                linewidth=1.8,
                linestyle="-",
                zorder=10
            )
        else:
            ax.plot(
                [p0[0], p1[0]],
                [p0[1], p1[1]],
                [p0[2], p1[2]],
                color="black",
                linewidth=1.2,
                linestyle="--",
                dashes=(4, 3),
                alpha=0.7,
                zorder=5
            )


def draw_plane(ax, polygon_pts, color):
    """
    画切面多边形
    """
    poly = Poly3DCollection(
        [polygon_pts],
        facecolors=color,
        edgecolors=color,
        linewidths=2.0,
        alpha=0.35
    )
    ax.add_collection3d(poly)


def draw_normal(ax, center, normal, color, length=0.35):
    """
    画双向法向箭头
    """
    n = normalize(normal)

    ax.quiver(
        center[0], center[1], center[2],
        n[0], n[1], n[2],
        length=length, normalize=True,
        color=color, linewidth=2.0, arrow_length_ratio=0.15
    )

    ax.quiver(
        center[0], center[1], center[2],
        -n[0], -n[1], -n[2],
        length=length, normalize=True,
        color=color, linewidth=2.0, arrow_length_ratio=0.15
    )


def style_ax(ax):
    """
    统一 3D 视图样式
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_zlim(0, 1)
    ax.set_box_aspect([1, 1, 1])

    # 固定视角，保证 13 张图一致
    ax.view_init(elev=22, azim=-60)

    # 去掉刻度
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    # 去掉坐标轴标签
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")

    # 去掉背景面板
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.grid(False)


def plot_single_direction(item, save_path):
    """
    画单张图
    """
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")

    polygon_pts = plane_cube_intersection(
        center=CUBE_CENTER,
        normal=item["normal"],
        vertices=VERTICES,
        edges=EDGES
    )

    draw_cube(ax)
    draw_plane(ax, polygon_pts, item["color"])
    draw_normal(ax, CUBE_CENTER, item["normal"], item["color"])

    style_ax(ax)

    normal_str = f"法向量: {item['normal']}"
    ax.set_title(f"{item['id']}. {item['name']}\n{normal_str}", fontsize=13, pad=18)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_overview(items, save_path, ncols=4):
    """
    画总览图
    """
    n = len(items)
    nrows = math.ceil(n / ncols)

    fig = plt.figure(figsize=(4.5 * ncols, 4.5 * nrows))

    for idx, item in enumerate(items, start=1):
        ax = fig.add_subplot(nrows, ncols, idx, projection="3d")

        polygon_pts = plane_cube_intersection(
            center=CUBE_CENTER,
            normal=item["normal"],
            vertices=VERTICES,
            edges=EDGES
        )

        draw_cube(ax)
        draw_plane(ax, polygon_pts, item["color"])
        draw_normal(ax, CUBE_CENTER, item["normal"], item["color"])

        style_ax(ax)
        ax.set_title(
            f"{item['id']}. {item['name']}\n法向量: {item['normal']}",
            fontsize=10,
            pad=10
        )

    plt.suptitle("立方体 13 个方向的切面示意图", fontsize=20, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def main():
    # 生成 13 张单图
    for item in DIRECTIONS:
        filename = f"{item['id']:02d}_{item['name'].replace('（', '_').replace('）', '').replace(' ', '_')}.png"
        filename = filename.replace("/", "_")
        save_path = os.path.join(OUTPUT_DIR, filename)
        plot_single_direction(item, save_path)
        print(f"已保存: {save_path}")

    # 生成总览图
    overview_path = os.path.join(OUTPUT_DIR, "00_立方体13个方向切面总览图.png")
    plot_overview(DIRECTIONS, overview_path)
    print(f"已保存总览图: {overview_path}")


if __name__ == "__main__":
    main()

/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 27178 (\N{CJK UNIFIED IDEOGRAPH-6A2A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 21521 (\N{CJK UNIFIED IDEOGRAPH-5411}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 19978 (\N{CJK UNIFIED IDEOGRAPH-4E0A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 19979 (\N{CJK UNIFIED IDEOGRAPH-4E0B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 26041 (\N{CJK UNIFIED IDEOGRAPH-65B9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 65289 (\N

已保存: /workspace/data/13_sections/01_横向_上下方向.png
已保存: /workspace/data/13_sections/02_矢状向_左右方向.png
已保存: /workspace/data/13_sections/03_冠状向_前后方向.png
已保存: /workspace/data/13_sections/04_体对角方向_1.png
已保存: /workspace/data/13_sections/05_体对角方向_2.png
已保存: /workspace/data/13_sections/06_体对角方向_3.png
已保存: /workspace/data/13_sections/07_体对角方向_4.png
已保存: /workspace/data/13_sections/08_面对角方向_1.png
已保存: /workspace/data/13_sections/09_面对角方向_2.png


/tmp/ipykernel_3016940/4276416634.py:290: UserWarning: Glyph 38754 (\N{CJK UNIFIED IDEOGRAPH-9762}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3016940/4276416634.py:291: UserWarning: Glyph 38754 (\N{CJK UNIFIED IDEOGRAPH-9762}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=200, bbox_inches="tight")


已保存: /workspace/data/13_sections/10_面对角方向_3.png
已保存: /workspace/data/13_sections/11_面对角方向_4.png
已保存: /workspace/data/13_sections/12_面对角方向_5.png
已保存: /workspace/data/13_sections/13_面对角方向_6.png


/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 27178 (\N{CJK UNIFIED IDEOGRAPH-6A2A}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 21521 (\N{CJK UNIFIED IDEOGRAPH-5411}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 19978 (\N{CJK UNIFIED IDEOGRAPH-4E0A}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 19979 (\N{CJK UNIFIED IDEOGRAPH-4E0B}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_3016940/4276416634.py:326: UserWarning: Glyph 26041 (\N{CJK UNIFIED IDEOGRAPH-65B9}) missing from font(s) Deja

已保存总览图: /workspace/data/13_sections/00_立方体13个方向切面总览图.png
